# 13 — Vanishing and exploding gradients

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/013-optimization-failure-modes.ipynb)

Backpropagation multiplies one factor per layer, so a gradient is a product. If each factor is a little below 1 the signal fades to nothing on the way back; if each is a little above 1 it blows up. This notebook measures that product, plots it on a log axis, lets you drag the depth, and then shows a training run that turns into `inf` and `nan`.

## 1. The numbers: ten layers, three gains

Call `g` the typical size of one layer's contribution to the gradient. Through `L` layers the gradient is multiplied by `g` exactly `L` times, so its size is `g` to the power `L`. Ten layers already separates the three cases below by six orders of magnitude.

In [ ]:
import numpy as np                                     # arrays and powers

depth = 10                                             # ten layers between the loss and the first weight

for g in (0.25, 1.0, 1.5):                             # three typical per-layer sizes
    grad = g ** depth                                  # the gradient after ten multiplications
    print(f"g={g:.2f}   g^10={grad:.3e}")              # 9.537e-07, 1.000e+00, 5.767e+01

**Line by line**

- `import numpy as np` — NumPy, already installed on Colab.
- `depth = 10` — how many layers the gradient must travel back through.
- `for g in (0.25, 1.0, 1.5):` — one pass for each per-layer gain.
- `grad = g ** depth` — the whole computation: one power.
- `print(f"...")` — three decimals for `g`, scientific notation for the result.

## 2. Step by step: multiply one layer at a time

The power is just repeated multiplication, so do it in a loop to see where the damage happens. Start at 1.0 at the loss and multiply by `g` once per layer, first with `g = 0.25` and then with `g = 1.5`, printing every fifth layer.

In [ ]:
g = 0.25                                               # a gain below 1: the vanishing case
grad = 1.0                                             # start at the loss with gradient size 1
for layer in range(1, 21):                             # walk backwards through 20 layers
    grad = grad * g                                    # one more layer, one more multiplication
    if layer % 5 == 0:                                 # report only every fifth layer
        print(f"g=0.25  after {layer:2d} layers: {grad:.3e}")   # 9.095e-13 at layer 20

g = 1.5                                                # a gain above 1: the exploding case
grad = 1.0                                             # reset to the loss
for layer in range(1, 21):                             # the same walk, different gain
    grad = grad * g                                    # each layer multiplies by 1.5
    if layer % 5 == 0:                                 # keep the printout short
        print(f"g=1.5   after {layer:2d} layers: {grad:.3e}")   # 3.325e+03 at layer 20

**Line by line**

- `g = 0.25` — a per-layer factor below one.
- `grad = 1.0` — the gradient arriving at the loss, set to one for easy reading.
- `for layer in range(1, 21):` — twenty backward steps, numbered from 1.
- `grad = grad * g` — each layer shrinks the gradient by a factor of four.
- `if layer % 5 == 0:` — only print rows 5, 10, 15, 20.
- `print(f"...")` — the shrinking value in scientific notation.
- `g = 1.5` — same experiment with a factor above one.
- `grad = 1.0` — reset the accumulator.
- `for layer in ...` / `grad = grad * g` — the identical loop, now growing.

## 3. Plot: gradient size against depth on a log axis

A linear axis would hide the vanishing case completely: 0.25 to the tenth is less than one millionth. A logarithmic y axis shows all three curves as straight lines, whose slopes are set by `g` itself.

In [ ]:
import matplotlib.pyplot as plt                        # the plotting library

depths = np.arange(1, 51)                              # one point per depth, 1 to 50

plt.figure(figsize=(7, 3.5))                           # a wide, short figure

for g, style in ((0.25, "o-"), (1.0, "s-"), (1.5, "^-")):   # one curve per gain
    plt.semilogy(depths, g ** depths, style, label=f"g = {g}")   # g^L for every depth

plt.axhline(1.0, color="grey", linestyle="--", linewidth=0.8)   # the healthy size
plt.xlabel("depth L (number of layers)")               # x axis: how far back we go
plt.ylabel("gradient magnitude g^L (log scale)")       # y axis: size of the signal
plt.legend()                                           # name the three curves
plt.title("below 1 it vanishes, above 1 it explodes")  # the takeaway
plt.show()                                             # render the figure

**Line by line**

- `import matplotlib.pyplot as plt` — the plotting library.
- `depths = np.arange(1, 51)` — the depths 1 through 50.
- `plt.figure(...)` — a wide figure so 50 points are readable.
- `for g, style in (...)` — pairs each gain with a line style.
- `plt.semilogy(...)` — base-10 log on the y axis, so straight lines mean constant ratios.
- `plt.axhline(1.0, ...)` — a dashed reference at the healthy gradient size.
- `plt.xlabel(...)` / `plt.ylabel(...)` — name both axes, including the log scale.
- `plt.title(...)` — the one-sentence takeaway.
- `plt.show()` — render it.

## 4. Interactive: drag the depth

The slider cuts the curves off at depth `L`. The faint lines show the full range for context; the solid dots show how far the gradient has travelled at the chosen depth. The title prints all three values, so you can find the depth where the exploding case passes one million.

In [ ]:
from ipywidgets import interact                        # sliders that rerun a function

def show(max_depth=20):                                # the slider supplies the depth
    L = np.arange(1, 51)                               # all depths, for the faint context lines
    plt.figure(figsize=(6.5, 3.5))                     # a fresh figure per drag
    for g, c in ((0.25, "tab:blue"), (1.0, "tab:green"), (1.5, "tab:red")):   # three gains
        vals = g ** L                                  # the gradient size at every depth
        plt.semilogy(L, vals, color=c, alpha=0.25)     # faint full curve
        plt.semilogy(L[:max_depth], vals[:max_depth], "o-", color=c, label=f"g = {g}")   # up to L
    plt.axvline(max_depth, color="grey", linestyle=":")             # mark the chosen depth
    plt.ylim(1e-9, 1e12)                               # keep the axes still while dragging
    plt.xlabel("depth L (number of layers)")           # x axis
    plt.ylabel("gradient magnitude g^L (log scale)")   # y axis
    plt.legend(loc="lower left")                       # name the curves
    plt.title(f"L={max_depth}:  0.25^L={0.25 ** max_depth:.1e}   "   # first half of the readout
              f"1.0^L=1.0   1.5^L={1.5 ** max_depth:.1e}")          # the readout
    plt.show()                                         # draw the frame

interact(show, max_depth=(1, 60, 1))                   # slider from 1 to 60 layers

**Line by line**

- `from ipywidgets import interact` — the widget helper.
- `def show(max_depth=20)` — redraws one frame at the chosen depth.
- `L = np.arange(1, 51)` — the depths used by the faint context lines.
- `for g, c in (...)` — each gain with its colour.
- `vals = g ** L` — the whole curve, computed once.
- `plt.semilogy(L, vals, alpha=0.25)` — the faint full curve behind the dots.
- `plt.semilogy(L[:max_depth], ...)` — only the first `max_depth` points are solid.
- `plt.axvline(...)` — a dotted marker at the slider's depth.
- `plt.ylim(1e-9, 1e12)` — fixed range so the plot does not jump between frames.
- `plt.title(f"...")` — live readout of all three gradient sizes.
- `interact(show, max_depth=(1, 60, 1))` — attach the slider.

## 5. When it breaks: 1.5^50 ≈ 6.4e8, then inf and nan

At depth 50 the exploding factor is about 640 million. Multiply that by a learning rate of 0.01 and the requested weight update is 6.4 million times the weight itself, so one step is enough to destroy the model. Run in 32-bit floats, the weight overflows to `inf` and then to `nan`, and every later step stays `nan`.

In [ ]:
w = np.float32(1.0)                                    # a weight, in the 32-bit floats GPUs use
G = np.float32(1.5) ** np.float32(50)                  # gradient size at depth 50: about 6.4e8
lr = np.float32(0.01)                                  # a learning rate that looks modest
print("gradient at depth 50 :", G)                     # 6.376e+08
print("requested step size  :", lr * G)                # 6.376e+06: millions of times the weight

history = []                                           # the weight after each step
with np.errstate(over="ignore", invalid="ignore"):     # let overflow happen so we can watch it
    for step in range(1, 9):                           # eight optimisation steps
        w = w - lr * G * w                             # gradient descent: w minus lr times gradient
        history.append(float(w))                       # record the value for the plot
        print(f"step {step}: w = {w}")                 # step 6 is inf, step 7 onward is nan

with np.errstate(over="ignore"):                       # we expect this overflow, so do not warn
    print("32-bit overflow: 1.5^250 =", np.float32(1.5) ** np.float32(250))   # inf

plt.figure(figsize=(7, 3.5))                           # plot the first six steps only
plt.semilogy(range(1, 7), [abs(v) for v in history[:6]], "o-", color="tab:red")   # |w| per step
plt.xlabel("optimisation step")                        # x axis
plt.ylabel("|weight| (log scale)")                     # y axis
plt.title("one exploding gradient: 1 -> 1e34 -> inf in six steps")   # the takeaway
plt.show()                                             # render the figure

print("fix: clip the step at 5, so it is 5.0 instead of", float(lr * G))   # clipping caps the damage

**Line by line**

- `w = np.float32(1.0)` — a single weight stored as a 32-bit float, as on a GPU.
- `G = ...` — the gradient size after 50 layers of factor 1.5.
- `lr = 0.01` — the learning rate.
- `print(...)` — shows the gradient and the step it asks for.
- `history = []` — records the weight after each step.
- `with np.errstate(...)` — silences the overflow warning so the run continues.
- `for step in range(1, 9):` — eight steps of gradient descent.
- `w = w - lr * G * w` — the update: subtract learning rate times gradient.
- `history.append(float(w))` — store the value for plotting.
- `print(...)` — prints every step, including `inf` and `nan`.
- `with np.errstate(over=...)` — a second silent block for the power below.
- `np.float32(1.5) ** np.float32(250)` — a second overflow, this time from the power alone.
- `plt.semilogy(...)` — the first six steps, before `inf`, on a log axis.
- `print("fix: ...")` — gradient clipping caps the step at 5 instead of millions.

## Your turn

Set the slider in section 4 to 50 and read all three values: which gain needs only 20 layers to pass 1000? Then change `lr` in section 5 to 0.0001 and find the step where `w` first becomes `inf` — did it delay the blow-up or prevent it?